# Change format of zoorvival datasets to be more uniform with the one from the original code

get the datasets and create new csv with a similar format

In [1]:
import os
import numpy as np
import pandas as pd
import copy

In [2]:
# get the whole data
study = 'BRCA'
direction = 'src/datasets_csv'

# clinical
clinical = pd.read_csv(
    os.path.join(f'{study}', f'{study}_clinical_preprocessed.csv'), 
    index_col=0
)

# omic 1 2 3 4
omic_cnv = pd.read_csv(
    os.path.join(f'{study}', f'{study}_CNV_filtered.csv'),    # all rna data
    engine='python', 
    index_col=0
)

omic_dna = pd.read_csv(
    os.path.join(f'{study}', f'{study}_DNAm_filtered.csv'),    # all rna data
    engine='python', 
    index_col=0
)

omic_mirna = pd.read_csv(
    os.path.join(f'{study}', f'{study}_miRNA_filtered.csv'),    # all rna data
    engine='python', 
    index_col=0
)

omic_mrna = pd.read_csv(
    os.path.join(f'{study}', f'{study}_mRNA_filtered.csv'),    # all rna data
    engine='python', 
    index_col=0
)

In [3]:
clinical.head(3)

,age,ajcc_metastasis_pathologic_pm,ajcc_nodes_pathologic_pn,ajcc_pathologic_tumor_stage,ajcc_staging_edition,ajcc_tumor_pathologic_pt,brachytherapy_total_dose_point_a,cent17_copy_number,days_to_initial_pathologic_diagnosis,dfs_time,...,sample_count,sex,site_of_tumor_tissue,staging_system,staging_system_other,surgery_for_positive_margins,surgery_for_positive_margins_other,surgical_procedure_first,tissue_source_site,tumor_status
patient_id,,,,,,,,,,,,,,,,,,,,,
4H-AAAK,50.0,M0,N2a,Stage IIIA,7th,T2,NaN,NaN,0.0,11.43,...,1,Female,Breast,Axillary lymph node dissection alone,NaN,NaN,NaN,Modified Radical Mastectomy,4H,TUMOR FREE
5L-AAT0,42.0,M0,N0,Stage IIA,7th,T2,NaN,NaN,0.0,48.52,...,1,Female,Breast,Axillary lymph node dissection alone,NaN,NaN,NaN,Modified Radical Mastectomy,5L,TUMOR FREE
5T-A9QA,52.0,MX,NX,Stage IIA,7th,T2,NaN,NaN,0.0,9.95,...,1,Female,Breast,No axillary staging,NaN,NaN,NaN,NaN,5T,NaN


start with clinical_data and label_data

patient_id --> case_id; consider to add TCGA- to the id

CLINICAL:

stage:
ajcc_pathologic_tumor_stage --> stage; from STAGE nstage to nstage

subtype:
add 'LUAC' everywhere

grade:
ajcc_staging_edition --> grade    (????)


LABEL:

case_id:
as above

slide_id:
the case_id is enought to find the slide_id, add some 'X' to have a similar format as the other data

age:
age --> age

site:
primary_site_patient --> site    (?????)

survival_months, survival_months_dss, survival_months_pfi:
dfs_time --> survival anything
os_time

censorship, censorship_dss, censorship_pfi:
dfs_event --> censorship anything    <<<<< need to change to 1- censorship??? >>>>>
os_event

\#\#\# disease-free survival (DFS); overall survival (OS); disease-specific survival(DSS), disease-free interval (DFI); progression-freeinterval (PFI)

is_female:
sex --> is_female;   Female/Male to 1.0/0.0

oncotree_code:
same as subtype in clinical_data

train:
???????

everything else can go into 'clinical_data', isn't going to be used

## LABEL_DATA

In [4]:
def to_case_id(patient):
    case_id = 'TCGA-' + patient
    return case_id

def to_slide_id(patient, other_patient_id):
    slide_id = 'TCGA-' + patient + '-XXX-XX-XXX.' + other_patient_id + '.svs'
    return slide_id

def to_is_female(sex):
    out = 0
    if sex == 'Female':
        out = 1
    return out

In [5]:
# create label_data
used_columns = []
# case_id, slide_id
label = pd.DataFrame()
label['case_id'] = clinical.index
label['case_id'] = to_case_id(label['case_id'].values)

label['slide_id'] = to_slide_id(clinical.index.values, clinical['other_patient_id'].values)

# age
label['age'] = clinical['age'].values
used_columns.append('age')

# site
where = 'tissue_source_site'
if study in ['LUAD', 'BLCA', 'BRCA']:
    where = 'primary_site_patient'
label['site'] = clinical[where].values
used_columns.append(where)

# censorship
label['censorship_dfs'] = 1- clinical['dfs_event'].values
label['censorship_os'] = 1- clinical['os_event'].values

label['survival_months_dfs'] = clinical['dfs_time'].values
label['survival_months_os'] = clinical['os_time'].values

used_columns.append('dfs_event')
used_columns.append('os_event')
used_columns.append('dfs_time')
used_columns.append('os_time')

# is_female
label['is_female'] = [to_is_female(x) for x in clinical['sex'].values]
used_columns.append('sex')

label['oncotree_code'] = f'{study}'

label['train'] = 1.0

print(label[:3])

        case_id                                           slide_id   age  \
0  TCGA-4H-AAAK  TCGA-4H-AAAK-XXX-XX-XXX.6623FC5E-00BE-4476-967...  50.0   
1  TCGA-5L-AAT0  TCGA-5L-AAT0-XXX-XX-XXX.86C6F993-327F-4525-998...  42.0   
2  TCGA-5T-A9QA  TCGA-5T-A9QA-XXX-XX-XXX.2FD36838-5A83-433E-AC8...  52.0   

                               site  censorship_dfs  censorship_os  \
0    Left|Left Upper Outer Quadrant             1.0              1   
1  Right|Right Lower Outer Quadrant             1.0              1   
2                              Left             1.0              1   

   survival_months_dfs  survival_months_os  is_female oncotree_code  train  
0                11.43               11.43          1          BRCA    1.0  
1                48.52               48.52          1          BRCA    1.0  
2                 9.95                9.95          1          BRCA    1.0  


In [6]:
# clean it a bit? eliminate the na?
for col in label.columns:
    print(col)
    df1 = label[label[col].isna()]
    print('num of missing rows: ', df1.shape[0])
    print(df1[col])

case_id
num of missing rows:  0
Series([], Name: case_id, dtype: object)
slide_id
num of missing rows:  0
Series([], Name: slide_id, dtype: object)
age
num of missing rows:  1
977   NaN
Name: age, dtype: float64
site
num of missing rows:  0
Series([], Name: site, dtype: object)
censorship_dfs
num of missing rows:  83
13     NaN
24     NaN
41     NaN
45     NaN
47     NaN
        ..
838    NaN
857    NaN
966    NaN
980    NaN
1004   NaN
Name: censorship_dfs, Length: 83, dtype: float64
censorship_os
num of missing rows:  0
Series([], Name: censorship_os, dtype: int64)
survival_months_dfs
num of missing rows:  84
13     NaN
24     NaN
41     NaN
45     NaN
47     NaN
        ..
857    NaN
966    NaN
977    NaN
980    NaN
1004   NaN
Name: survival_months_dfs, Length: 84, dtype: float64
survival_months_os
num of missing rows:  1
857   NaN
Name: survival_months_os, dtype: float64
is_female
num of missing rows:  0
Series([], Name: is_female, dtype: int64)
oncotree_code
num of missing rows:  0

there are 69 rows with missing dfs_censorship, 73 of missing survival_months. For os: 0 censorship, 8 for survival_months. Suggested to use os to calculate survival analysis

In [7]:
# remove na rows from os
print('before:') 
print(label.shape)
label = label.dropna(subset=['censorship_os', 'survival_months_os'])
print('after: ')
print(label.shape)
print(label[label['censorship_os'].isna()])
print(label[label['survival_months_os'].isna()])

if 'Unnamed: 0' in label.columns:
    label = label.drop(columns=['Unnamed: 0'])
    print('dropped')

before:
(1012, 11)
after: 
(1011, 11)
Empty DataFrame
Columns: [case_id, slide_id, age, site, censorship_dfs, censorship_os, survival_months_dfs, survival_months_os, is_female, oncotree_code, train]
Index: []
Empty DataFrame
Columns: [case_id, slide_id, age, site, censorship_dfs, censorship_os, survival_months_dfs, survival_months_os, is_female, oncotree_code, train]
Index: []


In [8]:
# save the new csv
label_folder = direction + f'/metadata/tcga_{study}.csv'
label.to_csv(f'{label_folder}')

## CLINICAL_DATA

In [9]:
def to_stage(stage):
    # remove 'Stage '
    stage = [str(txt).replace('Stage ', '') for txt in stage.values]
    return stage

In [10]:
clinical_data = pd.DataFrame()
clinical_data['case_id'] = to_case_id(clinical.index.values)

# stage
where = 'clinical_stage'
if study in ['LUAD', 'BLCA', 'BRCA']:
    where = 'ajcc_pathologic_tumor_stage'
clinical_data['stage'] = to_stage(clinical[where])
used_columns.append(where)

# subtype
clinical_data['subtype'] = f'{study}'

clinical_data['grade'] = clinical['ajcc_staging_edition'].values
used_columns.append('ajcc_staging_edition')

print(clinical_data[:3])

# add all the other columns
for col in clinical.columns:
    if col not in used_columns:
        clinical_data[col + '+'] = clinical[col].values

if 'Unnamed: 0' in clinical_data.columns:
    clinical_data = clinical_data.drop(columns=['Unnamed: 0'])
    print('dropped')
    
print(clinical_data[:3])

        case_id stage subtype grade
0  TCGA-4H-AAAK  IIIA    BRCA   7th
1  TCGA-5L-AAT0   IIA    BRCA   7th
2  TCGA-5T-A9QA   IIA    BRCA   7th
        case_id stage subtype grade ajcc_metastasis_pathologic_pm+  \
0  TCGA-4H-AAAK  IIIA    BRCA   7th                             M0   
1  TCGA-5L-AAT0   IIA    BRCA   7th                             M0   
2  TCGA-5T-A9QA   IIA    BRCA   7th                             MX   

  ajcc_nodes_pathologic_pn+ ajcc_tumor_pathologic_pt+  \
0                       N2a                        T2   
1                        N0                        T2   
2                        NX                        T2   

  brachytherapy_total_dose_point_a+ cent17_copy_number+  \
0                               NaN                 NaN   
1                               NaN                 NaN   
2                               NaN                 NaN   

   days_to_initial_pathologic_diagnosis+  ... retrospective_collection+  \
0                                 

In [11]:
# save the new csv
clinical_folder = direction + f'/clinical_data/tcga_{study}_clinical.csv'
clinical_data.to_csv(f'{clinical_folder}')

## OMICS DATA

the idea is to put them all together in a single csv, creating a 'signature' file where we include the attributes of each cathegory

In [12]:
def find_na(df):
    print('columns')
    num_col = 0
    for col in df.columns:
        df1 = df[df[col].isna()]
        if df1.shape[0] > 0:
            print(col)
            print('num of missing rows: ', df1.shape[0])
            print(df1[col])
            num_col += 1
    print('num of column with na: ', num_col)
    
    print('patients')
    nan_rows = df[df.isnull().T.any()]
    print('size: ', nan_rows.shape[0])
    print(nan_rows)
    

In [13]:
print('\nCNV')
find_na(omic_cnv)
print('\nDNA')
find_na(omic_dna)
print('\nmiRNA')
find_na(omic_mirna)
print('\nmRNA')
find_na(omic_mrna)


CNV
columns
num of column with na:  0
patients
size:  0
Empty DataFrame
Columns: [ENSG00000140798, ENSG00000261017, ENSG00000288026, ENSG00000260450, ENSG00000261538, ENSG00000261325, ENSG00000288601, ENSG00000261725, ENSG00000260889, ENSG00000261231, ENSG00000222268, ENSG00000262038, ENSG00000260497, ENSG00000121270, ENSG00000260782, ENSG00000155330, ENSG00000260281, ENSG00000222767, ENSG00000259983, ENSG00000259821, ENSG00000261173, ENSG00000069345, ENSG00000166123, ENSG00000140795, ENSG00000260086, ENSG00000260726, ENSG00000222170, ENSG00000260184, ENSG00000171208, ENSG00000259866, ENSG00000261336, ENSG00000259918, ENSG00000102910, ENSG00000279509, ENSG00000260744, ENSG00000261369, ENSG00000102893, ENSG00000260012, ENSG00000239840, ENSG00000129636, ENSG00000260347, ENSG00000275909, ENSG00000261802, ENSG00000260688, ENSG00000091651, ENSG00000069329, ENSG00000280376, ENSG00000261239, ENSG00000272545, ENSG00000260067, ENSG00000196470, ENSG00000171241, ENSG00000261131, ENSG00000260909,

In [14]:
# eliminate na columns
print('cnv before: ', omic_cnv.shape)
omic_cnv = omic_cnv.dropna(axis=1, how='any')
print('cnv after: ', omic_cnv.shape)

print('dna before: ', omic_dna.shape)
omic_dna = omic_dna.dropna(axis=1, how='any')
print('dna after: ', omic_dna.shape)

print('mirna before: ', omic_mirna.shape)
omic_mirna = omic_mirna.dropna(axis=1, how='any')
print('mirna after: ', omic_mirna.shape)

print('mrna before: ', omic_mrna.shape)
omic_mrna = omic_mrna.dropna(axis=1, how='any')
print('mrna after: ', omic_mrna.shape)

cnv before:  (1012, 4096)
cnv after:  (1012, 4096)
dna before:  (845, 4096)
dna after:  (845, 2404)
mirna before:  (1012, 306)
mirna after:  (1012, 191)
mrna before:  (1012, 4096)
mrna after:  (1012, 4096)


In [15]:
# check for na in dna data:
find_na(omic_dna)

columns
num of column with na:  0
patients
size:  0
Empty DataFrame
Columns: [cg22621695, cg25631352, cg08089301, cg12998491, cg12874092, cg06168050, cg04034767, cg04920951, cg07533148, cg14458834, cg08788717, cg17688525, cg16970232, cg13912117, cg18630040, cg04970117, cg02879662, cg26537639, cg18794577, cg03158400, cg22037648, cg02776251, cg13297865, cg12382902, cg26365854, cg11617144, cg21546671, cg14696396, cg11428724, cg26117023, cg15692239, cg23363832, cg09053680, cg16232979, cg18342279, cg27016990, cg14900471, cg17610929, cg08441170, cg11591325, cg08186362, cg21026663, cg25336579, cg01561916, cg23495733, cg21790626, cg07846220, cg13986130, cg05497616, cg06825142, cg14743462, cg24826867, cg23771603, cg26599006, cg20311501, cg00290506, cg27619475, cg08056146, cg09186006, cg07080358, cg04048259, cg14988503, cg11638200, cg20616414, cg06220235, cg04632671, cg19162106, cg03238797, cg14539231, cg19246110, cg26090652, cg02131967, cg14681055, cg09068528, cg08725962, cg00027083, cg13801416

In [16]:
# create the signatures
# some columns are duplicated, have the same name across multiple datasets, added a prefix to distinguish them
omic_cnv = omic_cnv.add_prefix('CNV_')
omic_dna = omic_dna.add_prefix('DNA_')
omic_mirna = omic_mirna.add_prefix('miRNA_')
omic_mrna = omic_mrna.add_prefix('mRNA')

sign = {'CNV': omic_cnv.columns.tolist(), 'DNA': omic_dna.columns.tolist(), 'miRNA': omic_mirna.columns.tolist(), 'mRNA': omic_mrna.columns.tolist() }
#print(sign)
max_len = max(len(v) for v in sign.values())
sign = {k: list(v) + [np.nan]*(max_len - len(v)) for k, v in sign.items()}
signatures = pd.DataFrame(sign)

signatures

,CNV,DNA,miRNA,mRNA
0,CNV_ENSG00000140798,DNA_cg22621695,miRNA_hsa-mir-196a-1,mRNAENSG00000110484
1,CNV_ENSG00000261017,DNA_cg25631352,miRNA_hsa-mir-205,mRNAENSG00000124935
2,CNV_ENSG00000288026,DNA_cg08089301,miRNA_hsa-mir-196a-2,mRNAENSG00000160182
3,CNV_ENSG00000260450,DNA_cg12998491,miRNA_hsa-mir-375,mRNAENSG00000159763
4,CNV_ENSG00000261538,DNA_cg12874092,miRNA_hsa-mir-9-2,mRNAENSG00000153002
...,...,...,...,...
4091,CNV_ENSG00000227413,NaN,NaN,mRNAENSG00000136160
4092,CNV_ENSG00000273176,NaN,NaN,mRNAENSG00000158186
4093,CNV_ENSG00000274280,NaN,NaN,mRNAENSG00000231367
4094,CNV_ENSG00000266320,NaN,NaN,mRNAENSG00000276557


In [17]:
#check for na values in dna again
find_na(omic_dna)

columns
num of column with na:  0
patients
size:  0
Empty DataFrame
Columns: [DNA_cg22621695, DNA_cg25631352, DNA_cg08089301, DNA_cg12998491, DNA_cg12874092, DNA_cg06168050, DNA_cg04034767, DNA_cg04920951, DNA_cg07533148, DNA_cg14458834, DNA_cg08788717, DNA_cg17688525, DNA_cg16970232, DNA_cg13912117, DNA_cg18630040, DNA_cg04970117, DNA_cg02879662, DNA_cg26537639, DNA_cg18794577, DNA_cg03158400, DNA_cg22037648, DNA_cg02776251, DNA_cg13297865, DNA_cg12382902, DNA_cg26365854, DNA_cg11617144, DNA_cg21546671, DNA_cg14696396, DNA_cg11428724, DNA_cg26117023, DNA_cg15692239, DNA_cg23363832, DNA_cg09053680, DNA_cg16232979, DNA_cg18342279, DNA_cg27016990, DNA_cg14900471, DNA_cg17610929, DNA_cg08441170, DNA_cg11591325, DNA_cg08186362, DNA_cg21026663, DNA_cg25336579, DNA_cg01561916, DNA_cg23495733, DNA_cg21790626, DNA_cg07846220, DNA_cg13986130, DNA_cg05497616, DNA_cg06825142, DNA_cg14743462, DNA_cg24826867, DNA_cg23771603, DNA_cg26599006, DNA_cg20311501, DNA_cg00290506, DNA_cg27619475, DNA_cg0805

In [18]:
# remove unnamed col if present
if 'Unnamed: 0' in signatures.columns:
    signatures = signatures.drop(columns=['Unnamed: 0'])
    print('dropped')

print(signatures.columns)

Index(['CNV', 'DNA', 'miRNA', 'mRNA'], dtype='object')


In [19]:
# save the new csv
sign_folder = direction + f'/metadata/{study}_signatures.csv'
signatures.to_csv(f'{sign_folder}')

now to the single omic file that contains all the omic data

In [20]:
find_na(omic_dna)

columns
num of column with na:  0
patients
size:  0
Empty DataFrame
Columns: [DNA_cg22621695, DNA_cg25631352, DNA_cg08089301, DNA_cg12998491, DNA_cg12874092, DNA_cg06168050, DNA_cg04034767, DNA_cg04920951, DNA_cg07533148, DNA_cg14458834, DNA_cg08788717, DNA_cg17688525, DNA_cg16970232, DNA_cg13912117, DNA_cg18630040, DNA_cg04970117, DNA_cg02879662, DNA_cg26537639, DNA_cg18794577, DNA_cg03158400, DNA_cg22037648, DNA_cg02776251, DNA_cg13297865, DNA_cg12382902, DNA_cg26365854, DNA_cg11617144, DNA_cg21546671, DNA_cg14696396, DNA_cg11428724, DNA_cg26117023, DNA_cg15692239, DNA_cg23363832, DNA_cg09053680, DNA_cg16232979, DNA_cg18342279, DNA_cg27016990, DNA_cg14900471, DNA_cg17610929, DNA_cg08441170, DNA_cg11591325, DNA_cg08186362, DNA_cg21026663, DNA_cg25336579, DNA_cg01561916, DNA_cg23495733, DNA_cg21790626, DNA_cg07846220, DNA_cg13986130, DNA_cg05497616, DNA_cg06825142, DNA_cg14743462, DNA_cg24826867, DNA_cg23771603, DNA_cg26599006, DNA_cg20311501, DNA_cg00290506, DNA_cg27619475, DNA_cg0805

In [21]:

combined_df = omic_dna.join([omic_cnv, omic_mirna, omic_mrna], how='outer')
combined_df[:3]

,DNA_cg22621695,DNA_cg25631352,DNA_cg08089301,DNA_cg12998491,DNA_cg12874092,DNA_cg06168050,DNA_cg04034767,DNA_cg04920951,DNA_cg07533148,DNA_cg14458834,...,mRNAENSG00000183840,mRNAENSG00000182463,mRNAENSG00000072571,mRNAENSG00000169607,mRNAENSG00000070526,mRNAENSG00000136160,mRNAENSG00000158186,mRNAENSG00000231367,mRNAENSG00000276557,mRNAENSG00000275791
3C-AALI,-3.529280,-1.283741,0.809936,-6.183612,-1.251846,-4.335636,-0.522545,0.211296,-3.474937,2.453824,...,2.904754,1.525016,3.363269,4.270110,1.723034,1.743989,3.610027,0.266037,1.643025,2.842657
3C-AALJ,-3.944544,-1.708141,0.805805,0.074889,-0.404185,-4.618542,1.127364,-0.344821,-5.519056,1.459287,...,2.121745,1.618709,4.345070,3.016514,0.651224,2.384050,4.342299,0.496820,2.081885,2.132478
3C-AALK,-3.093348,-1.532099,0.430924,-0.605442,-1.372814,-1.836377,-0.198532,-1.358223,-0.692899,0.853719,...,2.982145,2.221877,2.876095,2.212289,1.827697,2.520196,3.768692,0.446362,1.842335,1.937570


In [22]:
combined_df[combined_df.index == 'E9-A228']

,DNA_cg22621695,DNA_cg25631352,DNA_cg08089301,DNA_cg12998491,DNA_cg12874092,DNA_cg06168050,DNA_cg04034767,DNA_cg04920951,DNA_cg07533148,DNA_cg14458834,...,mRNAENSG00000183840,mRNAENSG00000182463,mRNAENSG00000072571,mRNAENSG00000169607,mRNAENSG00000070526,mRNAENSG00000136160,mRNAENSG00000158186,mRNAENSG00000231367,mRNAENSG00000276557,mRNAENSG00000275791
E9-A228,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.412131,1.601174,5.510886,3.780678,0.430071,3.233044,4.08203,0.204516,0.67717,0.188148


some patients in brca omic dna don't have any data for the dna attributes. Put 0s in their place and deal with it in the model

In [23]:
# checkk dna data
find_na(combined_df)

columns
DNA_cg22621695
num of missing rows:  167
E9-A228   NaN
LL-A5YP   NaN
OL-A66K   NaN
LL-A73Z   NaN
OL-A5RV   NaN
           ..
EW-A1OW   NaN
MS-A51U   NaN
E9-A54Y   NaN
LL-A6FQ   NaN
EW-A1PC   NaN
Name: DNA_cg22621695, Length: 167, dtype: float64
DNA_cg25631352
num of missing rows:  167
E9-A228   NaN
LL-A5YP   NaN
OL-A66K   NaN
LL-A73Z   NaN
OL-A5RV   NaN
           ..
EW-A1OW   NaN
MS-A51U   NaN
E9-A54Y   NaN
LL-A6FQ   NaN
EW-A1PC   NaN
Name: DNA_cg25631352, Length: 167, dtype: float64
DNA_cg08089301
num of missing rows:  167
E9-A228   NaN
LL-A5YP   NaN
OL-A66K   NaN
LL-A73Z   NaN
OL-A5RV   NaN
           ..
EW-A1OW   NaN
MS-A51U   NaN
E9-A54Y   NaN
LL-A6FQ   NaN
EW-A1PC   NaN
Name: DNA_cg08089301, Length: 167, dtype: float64
DNA_cg12998491
num of missing rows:  167
E9-A228   NaN
LL-A5YP   NaN
OL-A66K   NaN
LL-A73Z   NaN
OL-A5RV   NaN
           ..
EW-A1OW   NaN
MS-A51U   NaN
E9-A54Y   NaN
LL-A6FQ   NaN
EW-A1PC   NaN
Name: DNA_cg12998491, Length: 167, dtype: float64
DNA_cg128740

In [24]:
# find all na values in omic_dna and change the value from na to 0
combined_df = combined_df.fillna(0)

In [25]:
# change the index
combined_df.index = to_case_id(combined_df.index)
if 'Unnamed: 0' in combined_df.columns:
    combined_df = combined_df.drop(columns=['Unnamed: 0'])
    print('dropped')

print(combined_df.columns)
combined_df.head(3)

Index(['DNA_cg22621695', 'DNA_cg25631352', 'DNA_cg08089301', 'DNA_cg12998491',
       'DNA_cg12874092', 'DNA_cg06168050', 'DNA_cg04034767', 'DNA_cg04920951',
       'DNA_cg07533148', 'DNA_cg14458834',
       ...
       'mRNAENSG00000183840', 'mRNAENSG00000182463', 'mRNAENSG00000072571',
       'mRNAENSG00000169607', 'mRNAENSG00000070526', 'mRNAENSG00000136160',
       'mRNAENSG00000158186', 'mRNAENSG00000231367', 'mRNAENSG00000276557',
       'mRNAENSG00000275791'],
      dtype='object', length=10787)


,DNA_cg22621695,DNA_cg25631352,DNA_cg08089301,DNA_cg12998491,DNA_cg12874092,DNA_cg06168050,DNA_cg04034767,DNA_cg04920951,DNA_cg07533148,DNA_cg14458834,...,mRNAENSG00000183840,mRNAENSG00000182463,mRNAENSG00000072571,mRNAENSG00000169607,mRNAENSG00000070526,mRNAENSG00000136160,mRNAENSG00000158186,mRNAENSG00000231367,mRNAENSG00000276557,mRNAENSG00000275791
TCGA-3C-AALI,-3.529280,-1.283741,0.809936,-6.183612,-1.251846,-4.335636,-0.522545,0.211296,-3.474937,2.453824,...,2.904754,1.525016,3.363269,4.270110,1.723034,1.743989,3.610027,0.266037,1.643025,2.842657
TCGA-3C-AALJ,-3.944544,-1.708141,0.805805,0.074889,-0.404185,-4.618542,1.127364,-0.344821,-5.519056,1.459287,...,2.121745,1.618709,4.345070,3.016514,0.651224,2.384050,4.342299,0.496820,2.081885,2.132478
TCGA-3C-AALK,-3.093348,-1.532099,0.430924,-0.605442,-1.372814,-1.836377,-0.198532,-1.358223,-0.692899,0.853719,...,2.982145,2.221877,2.876095,2.212289,1.827697,2.520196,3.768692,0.446362,1.842335,1.937570


In [26]:
# check for na values
find_na(combined_df)

columns


num of column with na:  0
patients
size:  0
Empty DataFrame
Columns: [DNA_cg22621695, DNA_cg25631352, DNA_cg08089301, DNA_cg12998491, DNA_cg12874092, DNA_cg06168050, DNA_cg04034767, DNA_cg04920951, DNA_cg07533148, DNA_cg14458834, DNA_cg08788717, DNA_cg17688525, DNA_cg16970232, DNA_cg13912117, DNA_cg18630040, DNA_cg04970117, DNA_cg02879662, DNA_cg26537639, DNA_cg18794577, DNA_cg03158400, DNA_cg22037648, DNA_cg02776251, DNA_cg13297865, DNA_cg12382902, DNA_cg26365854, DNA_cg11617144, DNA_cg21546671, DNA_cg14696396, DNA_cg11428724, DNA_cg26117023, DNA_cg15692239, DNA_cg23363832, DNA_cg09053680, DNA_cg16232979, DNA_cg18342279, DNA_cg27016990, DNA_cg14900471, DNA_cg17610929, DNA_cg08441170, DNA_cg11591325, DNA_cg08186362, DNA_cg21026663, DNA_cg25336579, DNA_cg01561916, DNA_cg23495733, DNA_cg21790626, DNA_cg07846220, DNA_cg13986130, DNA_cg05497616, DNA_cg06825142, DNA_cg14743462, DNA_cg24826867, DNA_cg23771603, DNA_cg26599006, DNA_cg20311501, DNA_cg00290506, DNA_cg27619475, DNA_cg08056146, DN

In [27]:
# remove all na columns
#print('before: ', combined_df.shape)
#combined_df = combined_df.dropna(axis=1, how='any')
#print('after: ', combined_df.shape)

#find_na(combined_df)

In [28]:
omic_folder = direction + f'/raw_rna_data/other/{study}/rna_clean.csv'
combined_df.to_csv(f'{omic_folder}')